In [13]:
%pip install anthropic python-dotenv
%pip install requests ffmpeg-python
%pip install deepgram-sdk --upgrade
%pip install requests
%pip install anthropic

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [10]:
from dotenv import load_dotenv
load_dotenv("vars.env")

from anthropic import Anthropic

client = Anthropic()
model = "claude-haiku-4-5"

In [58]:
import re

def format_timestamp(seconds):
    hours = int(seconds // 3600)
    minutes = int((seconds % 3600) // 60)
    secs = int(seconds % 60)
    millis = int(round((seconds - int(seconds)) * 1000))
    return f"{hours:02d}:{minutes:02d}:{secs:02d},{millis:03d}"

def segments_to_srt(segments):
    lines = []
    for i, segment in enumerate(segments, start=1):
        start = format_timestamp(segment.start)
        end = format_timestamp(segment.end)
        text = segment.text.strip()
        lines.append(f"{i}\n{start} --> {end}\n{text}\n")
    return "\n".join(lines)

def _srt_content_to_text(content):
    lines = []
    for block in re.split(r"\n\s*\n", content.strip()):
        block_lines = block.strip().splitlines()
        # drop index line and timestamp line, keep the rest as spoken text
        for line in block_lines[2:] if len(block_lines) > 2 else block_lines[1:]:
            lines.append(line.strip())

    text = " ".join(lines)
    text = re.sub(r"\s+", " ", text).strip().lower()
    text = re.sub(r"[^\w\s]", "", text)
    return text

def _srt_to_text(srt_path):
    with open(srt_path, "r", encoding="utf-8") as f:
        content = f.read()
    return _srt_content_to_text(content)

def calculate_wer(reference_srt, hypothesis_srt):
    ref_words = _srt_to_text(reference_srt).split()
    hyp_words = _srt_to_text(hypothesis_srt).split()

    n, m = len(ref_words), len(hyp_words)
    dp = [[0] * (m + 1) for _ in range(n + 1)]
    for i in range(n + 1):
        dp[i][0] = i
    for j in range(m + 1):
        dp[0][j] = j

    for i in range(1, n + 1):
        for j in range(1, m + 1):
            if ref_words[i - 1] == hyp_words[j - 1]:
                dp[i][j] = dp[i - 1][j - 1]
            else:
                dp[i][j] = 1 + min(dp[i - 1][j], dp[i][j - 1], dp[i - 1][j - 1])

    edits = dp[n][m]
    return edits / n if n > 0 else 0.0

In [48]:
import subprocess
import os
import glob
from openai import OpenAI

client = OpenAI()

def whisper1_transcribe(mp3file, prompt):
    with open(mp3file, "rb") as audio_file:
        transcription = client.audio.transcriptions.create(
            model="whisper-1",
            file=audio_file,
            prompt=prompt,
            response_format="srt",
        )
    return transcription

def whisper_other_transcribe(mp3file, model, prompt):
    client = OpenAI(api_key=os.getenv("GROQ_API_KEY"), base_url="https://api.groq.com/openai/v1")

    r = client.audio.transcriptions.create(
        file=open(mp3file, "rb"),
        model=model,
        prompt=prompt,
        response_format="verbose_json",
        timestamp_granularities=["segment", "word"],
        language="en",
    )
    return segments_to_srt(r.segments)

#with open("it only takes one night… - champ kent (1080p).srt", "w") as srt_file:
#    srt_file.write(transcription)


#srt_text = segments_to_srt(r.segments)
#with open("whisper-v3-LARGEchampkent.srt", "w", encoding="utf-8") as srt_file:
    #srt_file.write(srt_text)

In [16]:
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages, system=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
    }
    if system:
        params["system"] = system
            
    message = client.messages.create(**params)
        
    # gets handled this way because we need to pick out the text block, not just content[0], since thinking blocks need to be handled
    return "".join(b.text for b in message.content if b.type == "text")

messages = []

while True:
    user_input = input("> ")
    if not user_input.strip():
        break
    add_user_message(messages, user_input)
    answer = chat(messages, system=system_prompt)
    add_assistant_message(messages, answer)
    print("---")
    print(answer)
    print("---")

In [69]:
def evaluate_model_on_golden_set(golden_set_dir="golden_set", prompt="", model="whisper-1"):
    scores = {}

    ## code grading - transcript WER. text only no timestamps.
    for entry in sorted(os.listdir(golden_set_dir)):
        folder = os.path.join(golden_set_dir, entry)
        if not os.path.isdir(folder):
            continue

        reference_srt = os.path.join(folder, "corrected_transcript.srt")
        if not os.path.exists(reference_srt):
            continue

        mp3_files = glob.glob(os.path.join(folder, "*.mp3"))
        if not mp3_files:
            continue
        mp3_file = mp3_files[0]

        if model == "whisper-1":
            transcription = whisper1_transcribe(mp3_file, prompt)
        else:
            transcription = whisper_other_transcribe(mp3_file, model, prompt)

        hypothesis_srt = os.path.join(folder, f"{model}_transcript.srt")
        with open(hypothesis_srt, "w", encoding="utf-8") as f:
            f.write(transcription)

        #print(_srt_to_text(reference_srt))
        #print(_srt_to_text(hypothesis_srt))


        scores[entry] = calculate_wer(reference_srt, hypothesis_srt)

    mean_errorscore = sum(scores.values()) / len(scores) if scores else 0.0

    return mean_errorscore, scores


prompt = "the channel is named Defence"

models_to_test = ["whisper-1", "whisper-large-v3", "whisper-large-v3-turbo"]
mean_scores_by_model = {}

for model_name in models_to_test:
    mean_errorscore, scores_by_video = evaluate_model_on_golden_set("golden_set", prompt, model_name)
    #print(scores_by_video)
    #print(mean_errorscore)
    mean_scores_by_model[model_name] = mean_errorscore

print("Using prompt: ", prompt)

print("Models tested: " + ", ".join(f"{name}, WER Score: {score}" for name, score in mean_scores_by_model.items()))

RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `whisper-large-v3` in organization `org_01m08rpjf1ebyb3cvpg5f8wwvp` service tier `on_demand` on seconds of audio per hour (ASPH): Limit 7200, Used 7139, Requested 502. Please try again in 3m40.5s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'seconds', 'code': 'rate_limit_exceeded'}}